<a href="https://colab.research.google.com/github/ShubhangiDimri/Indic-Meme-Understanding-Sentiment-Analysis-IMUSA-/blob/main/notebooks/IMUSA_MuRIL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

folder = "/content/drive/MyDrive/IMUSA"

print(os.listdir(folder))

['split_train.csv', 'split_val.csv']


In [6]:
import pandas as pd

folder = "/content/drive/MyDrive/IMUSA"

train_split = pd.read_csv(f"{folder}/split_train.csv")
val_split = pd.read_csv(f"{folder}/split_val.csv")

print("Training split shape:", train_split.shape)
print("Validation split shape:", val_split.shape)

print("\nTraining columns:")
print(train_split.columns.tolist())

print("\nValidation columns:")
print(val_split.columns.tolist())

display(train_split.head())
display(val_split.head())

Training split shape: (2402, 3)
Validation split shape: (600, 3)

Training columns:
['Id', 'Category', 'Text']

Validation columns:
['Id', 'Category', 'Text']


,Id,Category,Text
0,image_punjabi_619.jpg,Motivational,ਹੰਕਾਰ ਇਨਸਾਨ ਨੂੰ ਲੈ ਡੁੱਬਦਾ। ਬੰਦਿਆਂ ਤੂੰ ਮਾਣ ਨਾ ਕ...
1,image_punjabi_3382.jpg,Neutral,ਮੈਂ ਨੀ ਤੇਰੇ ਨਾਲ ਰਹਿਣਾ ਤੂੰ ਮੈਨੂੰ ਰੁਵਾਉਂਦਾ ਰਹਿਨਾ
2,image_punjabi_3445.jpg,Neutral,ਕਈ ਵੇਲੇ Situation ਏਦਾ ਦੀ ਹੋ ਜਾਦੀ ਕਿ ਸਮਝ ਨੀ ਲੱਗ...
3,image_punjabi_1051.jpg,Neutral,ਇਹ ਹਨ ਉਹ ਮਹਾਨ ਪੱਤਰਕਾਰ ਸ੍ਰੀ ਵਿਸ਼ਵਾਮਿੱਤਰ ਟੰਡਨ ਜੀ ...
4,image_punjabi_2953.jpg,Sarcasm,ਜਦੋ ਕੋਈ ਕਹਿੰਦਾ : ਤੇਰੀਆਂ ਅੱਖਾ ਵਿੱਚ ਡੁੱਬ ਜਾਣ ਨੂੰ...


,Id,Category,Text
0,image_punjabi_1769.jpg,Neutral,ਸਮਝਣਾਆਸਾਨ ਨਹੀ ਹੈ ਉਸ ਇਨਸਾਨ ਨੂੰ ਸਮਝਣਾ ਜੋ ਜਾਣਦਾ ਸ...
1,image_punjabi_2590.jpg,Motivational,"ਮਿਲੇ ਤਾਂ ਹਜਾਰਾਂ ਲੋਕ ਸੀ, ਜ਼ਿੰਦਗੀ ਚ' ਪਰ ਉਹ ਸਭ ਤੋ..."
2,image_punjabi_1685.jpg,Motivational,ਭਰੋਸਾ ਰਬੜ ਦੇ ਵਾਂਗੂ ਹੁੰਦਾ ਹੈ...ਜੋ ਹਰ ਗਲਤੀ ਦੇ ਬਾ...
3,image_punjabi_1404.jpg,Sarcasm,"""ਇੱਕ ਨੋਬਲ ਪੁਰਸਕਾਰ"" ਉਨ੍ਹਾਂ ਕੁੜੀਆਂ ਨੂੰ ਵੀ ਦੇਣਾ ਚ..."
4,image_punjabi_753.jpg,Neutral,ਮੈਂ ਨੀ ਚਾਹੁੰਦੀ ਹਮੇਸ਼ਾ ਮੇਰੇ ਕੋਲ ਰਹੇ ਬਸ ਜਿਥੇ ਵੀ ...


In [7]:
!pip install -q transformers sentencepiece accelerate scikit-learn

In [8]:
import torch
import transformers
import sklearn

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

Transformers: 5.13.1
PyTorch: 2.11.0+cu128
GPU: Tesla T4


In [9]:
print("Exact labels:")
print(sorted(train_split["Category"].dropna().unique()))

print("\nClass distribution:")
print(train_split["Category"].value_counts())

Exact labels:
['Motivational', 'Neutral', 'Offensive', 'Sarcasm']

Class distribution:
Category
Sarcasm         1063
Motivational     696
Neutral          601
Offensive         42
Name: count, dtype: int64


In [10]:
print("Validation class distribution:")
print(val_split["Category"].value_counts())

print("\nValidation percentages:")
print((val_split["Category"].value_counts(normalize=True) * 100).round(2))

Validation class distribution:
Category
Sarcasm         266
Motivational    174
Neutral         150
Offensive        10
Name: count, dtype: int64

Validation percentages:
Category
Sarcasm         44.33
Motivational    29.00
Neutral         25.00
Offensive        1.67
Name: proportion, dtype: float64


In [11]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

# Fixed deterministic label mapping
labels = sorted(train_split["Category"].unique())

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print("Label mapping:")
print(label2id)

# Compute balanced class weights from TRAINING DATA ONLY
classes = np.array(labels)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_split["Category"]
)

class_weights = torch.tensor(weights, dtype=torch.float32)

print("\nClass weights:")
for label, weight in zip(labels, weights):
    print(f"{label:15s}: {weight:.4f}")

print("\nWeights tensor:", class_weights)

Label mapping:
{'Motivational': 0, 'Neutral': 1, 'Offensive': 2, 'Sarcasm': 3}

Class weights:
Motivational   : 0.8628
Neutral        : 0.9992
Offensive      : 14.2976
Sarcasm        : 0.5649

Weights tensor: tensor([ 0.8628,  0.9992, 14.2976,  0.5649])


In [12]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "google/muril-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label
)

print("MuRIL loaded successfully.")

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  953MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

MuRIL loaded successfully.


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Using:", device)
print("Model device:", next(model.parameters()).device)

Using: cuda
Model device: cuda:0


In [14]:
from torch.utils.data import Dataset, DataLoader

MAX_LENGTH = 128

class IMUSATextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, label2id, max_length=128):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = str(row["Text"]) if pd.notna(row["Text"]) else ""
        label = self.label2id[row["Category"]]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.long)
        }


train_dataset = IMUSATextDataset(
    train_split,
    tokenizer,
    label2id,
    MAX_LENGTH
)

val_dataset = IMUSATextDataset(
    val_split,
    tokenizer,
    label2id,
    MAX_LENGTH
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

Train dataset: 2402
Validation dataset: 600


In [15]:
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

batch = next(iter(train_loader))

print("input_ids:", batch["input_ids"].shape)
print("attention_mask:", batch["attention_mask"].shape)
print("labels:", batch["label"].shape)

input_ids: torch.Size([16, 128])
attention_mask: torch.Size([16, 128])
labels: torch.Size([16])


In [16]:
from torch.optim import AdamW
import torch.nn as nn

# Optimizer
LEARNING_RATE = 2e-5

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

# Weighted loss for class imbalance
loss_fn = nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", LEARNING_RATE)
print("Class weights:", class_weights)

Optimizer: AdamW
Learning rate: 2e-05
Class weights: tensor([ 0.8628,  0.9992, 14.2976,  0.5649])


In [17]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

EPOCHS = 4

best_macro_f1 = -1
best_state = None
training_history = []

for epoch in range(EPOCHS):

    # -------------------------
    # TRAIN
    # -------------------------
    model.train()

    total_train_loss = 0

    train_progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS} - Training"
    )

    for batch in train_progress:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["label"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_fn(outputs.logits, labels_batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_train_loss += loss.item()

        train_progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    avg_train_loss = total_train_loss / len(train_loader)

    # -------------------------
    # VALIDATION
    # -------------------------
    model.eval()

    total_val_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():

        val_progress = tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{EPOCHS} - Validation"
        )

        for batch in val_progress:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_batch = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = loss_fn(
                outputs.logits,
                labels_batch
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            all_preds.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                labels_batch.cpu().numpy()
            )

    avg_val_loss = total_val_loss / len(val_loader)

    # -------------------------
    # METRICS
    # -------------------------
    accuracy = accuracy_score(
        all_labels,
        all_preds
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )[2]

    cm = confusion_matrix(
        all_labels,
        all_preds,
        labels=list(range(len(labels)))
    )

    print("\n" + "=" * 60)
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss:      {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Accuracy:        {accuracy:.4f}")
    print(f"Macro Precision: {precision:.4f}")
    print(f"Macro Recall:    {recall:.4f}")
    print(f"Macro F1:        {f1:.4f}")
    print(f"Weighted F1:     {weighted_f1:.4f}")
    print("Confusion Matrix:")
    print(cm)
    print("=" * 60)

    training_history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "accuracy": accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "weighted_f1": weighted_f1
    })

    # Save best model in memory
    if f1 > best_macro_f1:

        best_macro_f1 = f1

        best_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }

        print(f"*** New best Macro-F1: {best_macro_f1:.4f} ***")

print("\nTraining complete.")
print("Best Validation Macro-F1:", best_macro_f1)

Epoch 1/4 - Training:   0%|          | 0/151 [00:00<?, ?it/s]

Epoch 1/4 - Validation:   0%|          | 0/38 [00:00<?, ?it/s]


Epoch 1/4
Train Loss:      1.3711
Validation Loss: 1.3632
Accuracy:        0.4667
Macro Precision: 0.4251
Macro Recall:    0.2737
Macro F1:        0.2057
Weighted F1:     0.3315
Confusion Matrix:
[[  8  15   0 151]
 [  0   9   0 141]
 [  0   0   0  10]
 [  1   2   0 263]]
*** New best Macro-F1: 0.2057 ***


Epoch 2/4 - Training:   0%|          | 0/151 [00:00<?, ?it/s]

Epoch 2/4 - Validation:   0%|          | 0/38 [00:00<?, ?it/s]

...

Epoch 3/4 - Validation:   0%|          | 0/38 [00:00<?, ?it/s]


Epoch 3/4
Train Loss:      1.1835
Validation Loss: 1.1779
Accuracy:        0.6133
Macro Precision: 0.4287
Macro Recall:    0.4460
Macro F1:        0.4257
Weighted F1:     0.5918
Confusion Matrix:
[[146  19   0   9]
 [ 67  38   0  45]
 [  2   1   0   7]
 [ 46  36   0 184]]
*** New best Macro-F1: 0.4257 ***


Epoch 4/4 - Training:   0%|          | 0/151 [00:00<?, ?it/s]

Epoch 4/4 - Validation:   0%|          | 0/38 [00:00<?, ?it/s]


Epoch 4/4
Train Loss:      1.0611
Validation Loss: 1.1404
Accuracy:        0.6267
Macro Precision: 0.6966
Macro Recall:    0.5040
Macro F1:        0.5336
Weighted F1:     0.6235
Confusion Matrix:
[[120  39   0  15]
 [ 35  59   0  56]
 [  1   3   2   4]
 [ 25  46   0 195]]
*** New best Macro-F1: 0.5336 ***

Training complete.
Best Validation Macro-F1: 0.5335770022416018


In [18]:
import os
import torch

SAVE_DIR = "/content/drive/MyDrive/IMUSA/results/text_muril"
os.makedirs(SAVE_DIR, exist_ok=True)

# Restore best model
model.load_state_dict(best_state)
model = model.to(device)

# Save checkpoint
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "label2id": label2id,
        "id2label": id2label,
        "best_macro_f1": best_macro_f1
    },
    f"{SAVE_DIR}/best_model.pt"
)

# Save training history
history_df = pd.DataFrame(training_history)
history_df.to_csv(
    f"{SAVE_DIR}/training_log.csv",
    index=False
)

print("Saved to:", SAVE_DIR)
print("Best Macro-F1:", best_macro_f1)

Saved to: /content/drive/MyDrive/IMUSA/results/text_muril
Best Macro-F1: 0.5335770022416018


In [19]:
from sklearn.metrics import classification_report
import torch.nn.functional as F

model.eval()

all_probs = []
all_preds = []
all_labels = []
all_ids = []

with torch.no_grad():
    for batch_idx, batch in enumerate(val_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        probs = F.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels_batch.cpu().numpy())

probs_array = np.array(all_probs)
preds_array = np.array(all_preds)
labels_array = np.array(all_labels)

val_results = val_split.copy()

val_results["true_label"] = val_results["Category"]
val_results["predicted_id"] = preds_array
val_results["predicted_label"] = [
    id2label[i] for i in preds_array
]

for i, label in id2label.items():
    val_results[f"prob_{label}"] = probs_array[:, i]

val_results.to_csv(
    f"{SAVE_DIR}/val_predictions.csv",
    index=False
)

print("Saved validation predictions.")
print(f"Best Macro-F1: {best_macro_f1:.4f}")

Saved validation predictions.
Best Macro-F1: 0.5336
